# Used-car price prediction

This notebook explores the cleaned Car Price Assignment dataset (Data 3). The cleaning step is reproducible in src/preprocess_data3.py. Optional Data 1 and Data 2 files are loaded when available for later comparison.

In [2]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
def first_existing(paths):
    for path in paths:
        candidate = Path(path)
        if candidate.exists():
            return candidate
    raise FileNotFoundError("None of these dataset paths exists:\n" + "\n".join(map(str, paths)))


def optional_csv(paths):
    for path in paths:
        candidate = Path(path)
        if candidate.exists():
            return pd.read_csv(candidate)
    return None


df_3_path = first_existing([
    "data/processed/data3_cleaned.csv",
    "../data/processed/data3_cleaned.csv",
    "/kaggle/input/datasets/mazenbassem/resources/data/processed/data3_cleaned.csv",
    "/kaggle/input/car-pricev3/data3_cleaned.csv",
])
df_3 = pd.read_csv(df_3_path)
df_1 = optional_csv([
    "data/processed/data1_cleaned.csv",
    "../data/processed/data1_cleaned.csv",
    "/kaggle/input/datasets/mazenbassem/resources/data/processed/data1_cleaned.csv",
])
df_2 = optional_csv([
    "data/processed/data2_cleaned.csv",
    "../data/processed/data2_cleaned.csv",
    "/kaggle/input/datasets/mazenbassem/resources/data/processed/data2_cleaned.csv",
])
print(f"Data 3 loaded from: {df_3_path}")
print(f"Data 3 shape: {df_3.shape}")

In [ ]:
display(df_3.head())
print("Duplicate rows:", df_3.duplicated().sum())
print("Missing values:")
display(df_3.isna().sum().sort_values(ascending=False).to_frame("missing"))
display(df_3.describe(include="all").T)

In [ ]:
# Price and vehicle-size distributions
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.histplot(df_3["price"], kde=True, ax=axes[0])
axes[0].set_title("Price distribution")
sns.scatterplot(data=df_3, x="horsepower", y="price", hue="fuel_type", ax=axes[1])
axes[1].set_title("Price vs horsepower")
sns.scatterplot(data=df_3, x="curb_weight", y="price", hue="body_style", ax=axes[2])
axes[2].set_title("Price vs curb weight")
plt.tight_layout()

In [ ]:
numeric_columns = [
    "symboling", "wheelbase", "car_length", "car_width", "car_height",
    "curb_weight", "engine_size", "bore_ratio", "stroke",
    "compression_ratio", "horsepower", "peak_rpm", "city_mpg", "highway_mpg",
]
categorical_columns = [
    "brand", "fuel_type", "aspiration", "doors", "body_style", "drive_wheel",
    "engine_location", "engine_type", "cylinders", "fuel_system",
]
X = df_3[numeric_columns + categorical_columns]
y = df_3["price"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

numeric_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])
preprocessor = ColumnTransformer([
    ("numeric", numeric_preprocessor, numeric_columns),
    ("categorical", categorical_preprocessor, categorical_columns),
])

In [ ]:
models = {
    "Linear regression": LinearRegression(),
    "Random forest": RandomForestRegressor(
        n_estimators=300, random_state=RANDOM_STATE, min_samples_leaf=2, n_jobs=-1
    ),
}
results = []
for name, estimator in models.items():
    pipeline = Pipeline([("preprocessor", preprocessor), ("model", estimator)])
    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)
    results.append({
        "model": name,
        "MAE": mean_absolute_error(y_test, predictions),
        "RMSE": mean_squared_error(y_test, predictions, squared=False),
        "R2": r2_score(y_test, predictions),
    })

results_df = pd.DataFrame(results).sort_values("RMSE")
display(results_df)

In [ ]:
best_model_name = results_df.iloc[0]["model"]
best_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", models[best_model_name]),
])
best_pipeline.fit(X_train, y_train)

if best_model_name == "Random forest":
    feature_names = best_pipeline.named_steps["preprocessor"].get_feature_names_out()
    importances = pd.Series(
        best_pipeline.named_steps["model"].feature_importances_, index=feature_names
    ).sort_values(ascending=False).head(15)
    display(importances.to_frame("importance"))
    importances.sort_values().plot.barh(figsize=(8, 6), title="Top model features")
    plt.tight_layout()

### Notes

- `car_ID` is retained in the cleaned export for traceability but excluded from modeling.
- `CarName` is split into normalized `brand` and `model`; the high-cardinality model field is not used in this baseline.
- Preprocessing is fitted only on the training partition through a scikit-learn pipeline.
- Metrics are a baseline comparison, not a final claim about performance; use cross-validation and tuned models in the modeling step.